In [ ]:
!find /kaggle/input/datasets -maxdepth 5

In [ ]:
import shutil, os

SOURCE_PATH = "/kaggle/input/datasets/ilikesomeone/adas-project-2/adas_project 2"

if os.path.exists("/kaggle/working/adas_project"):
    shutil.rmtree("/kaggle/working/adas_project")

shutil.copytree(SOURCE_PATH, "/kaggle/working/adas_project")
%cd /kaggle/working/adas_project
!ls scripts

In [ ]:
%%writefile /kaggle/working/adas_project/data/phase1_dataset.py
"""
Phase 1 dataset: images + boxes only, no drivable-area/weather heads.
"""
import random
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset


class Phase1Dataset(Dataset):
    def __init__(self, merged_root, split="train", img_size=640, augment=None):
        self.root = Path(merged_root)
        self.img_dir = self.root / "images" / split
        self.lbl_dir = self.root / "labels" / split
        self.img_size = img_size
        self.augment = augment if augment is not None else (split == "train")
        self.image_files = sorted(
            list(self.img_dir.glob("*.png"))
            + list(self.img_dir.glob("*.ppm"))
            + list(self.img_dir.glob("*.jpg"))
            + list(self.img_dir.glob("*.jpeg"))
        )

    def __len__(self):
        return len(self.image_files)

    def _load_boxes(self, stem):
        txt = self.lbl_dir / f"{stem}.txt"
        boxes, classes = [], []
        if txt.exists() and txt.stat().st_size > 0:
            for line in txt.read_text().strip().splitlines():
                c, cx, cy, w, h = map(float, line.split())
                boxes.append([cx, cy, w, h])
                classes.append(int(c))
        return np.array(boxes, dtype=np.float32), np.array(classes, dtype=np.int64)

    def _apply_augmentation(self, img, boxes_norm):
        if random.random() < 0.5:
            img = cv2.flip(img, 1)
            if len(boxes_norm):
                boxes_norm = boxes_norm.copy()
                boxes_norm[:, 0] = 1.0 - boxes_norm[:, 0]

        if random.random() < 0.5:
            alpha = random.uniform(0.8, 1.2)
            beta = random.uniform(-15, 15)
            img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

        return img, boxes_norm

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        stem = img_path.stem
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))

        boxes_norm, classes = self._load_boxes(stem)

        if self.augment:
            img, boxes_norm = self._apply_augmentation(img, boxes_norm)

        img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        if len(boxes_norm):
            cx, cy, bw, bh = boxes_norm[:, 0], boxes_norm[:, 1], boxes_norm[:, 2], boxes_norm[:, 3]
            x1 = (cx - bw / 2) * self.img_size
            y1 = (cy - bh / 2) * self.img_size
            x2 = (cx + bw / 2) * self.img_size
            y2 = (cy + bh / 2) * self.img_size
            boxes_xyxy = np.stack([x1, y1, x2, y2], axis=1).astype(np.float32)
        else:
            boxes_xyxy = np.zeros((0, 4), dtype=np.float32)

        return {
            "image": img_t,
            "boxes": torch.from_numpy(boxes_xyxy),
            "classes": torch.from_numpy(classes),
            "file": img_path.name,
        }


def collate_fn(batch):
    images = torch.stack([b["image"] for b in batch])
    boxes = [b["boxes"] for b in batch]
    classes = [b["classes"] for b in batch]
    files = [b["file"] for b in batch]
    return {"image": images, "boxes": boxes, "classes": classes, "file": files}

In [ ]:
%%writefile /kaggle/working/adas_project/models/losses.py
"""
Loss functions for the hybrid detector.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def box_iou(boxes1, boxes2):
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(0)
    lt = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp(0)
    inter = wh[..., 0] * wh[..., 1]
    union = area1[:, None] + area2[None, :] - inter
    return inter / union.clamp(min=1e-7)


def ciou_loss(pred, target):
    iou = box_iou(pred, target).diag()
    px1, py1, px2, py2 = pred.unbind(-1)
    tx1, ty1, tx2, ty2 = target.unbind(-1)
    pcx, pcy = (px1 + px2) / 2, (py1 + py2) / 2
    tcx, tcy = (tx1 + tx2) / 2, (ty1 + ty2) / 2
    center_dist = (pcx - tcx) ** 2 + (pcy - tcy) ** 2

    ex1, ey1 = torch.min(px1, tx1), torch.min(py1, ty1)
    ex2, ey2 = torch.max(px2, tx2), torch.max(py2, ty2)
    diag = (ex2 - ex1) ** 2 + (ey2 - ey1) ** 2 + 1e-7

    pw, ph = (px2 - px1).clamp(min=1e-6), (py2 - py1).clamp(min=1e-6)
    tw, th = (tx2 - tx1).clamp(min=1e-6), (ty2 - ty1).clamp(min=1e-6)
    v = (4 / (torch.pi ** 2)) * (torch.atan(tw / th) - torch.atan(pw / ph)) ** 2
    with torch.no_grad():
        alpha = v / (1 - iou + v + 1e-7)

    ciou = iou - center_dist / diag - alpha * v
    return 1 - ciou


def assign_targets_simple(pred_boxes, anchors, gt_boxes, gt_classes, num_classes,
                           center_radius=2.5, stride_tensor=None):
    N = anchors.shape[0]
    device = anchors.device
    target_cls = torch.full((N,), -1, dtype=torch.long, device=device)
    target_box = torch.zeros((N, 4), dtype=torch.float32, device=device)

    if gt_boxes.numel() == 0:
        return target_cls, target_box, torch.zeros(N, dtype=torch.bool, device=device)

    gt_cx = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
    gt_cy = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2

    for g in range(gt_boxes.shape[0]):
        radius_px = center_radius * stride_tensor.squeeze(-1)
        dist = ((anchors[:, 0] - gt_cx[g]) ** 2 + (anchors[:, 1] - gt_cy[g]) ** 2).sqrt()
        candidates = dist < radius_px
        if candidates.sum() == 0:
            continue
        ious = box_iou(pred_boxes[candidates], gt_boxes[g:g + 1]).squeeze(-1)
        cand_idx = candidates.nonzero(as_tuple=True)[0]
        best_local = ious.argmax()
        best_idx = cand_idx[best_local]
        target_cls[best_idx] = gt_classes[g]
        target_box[best_idx] = gt_boxes[g]

        k = min(3, candidates.sum().item())
        topk_idx = cand_idx[ious.topk(k).indices]
        target_cls[topk_idx] = gt_classes[g]
        target_box[topk_idx] = gt_boxes[g]

    pos_mask = target_cls >= 0
    return target_cls, target_box, pos_mask


class DetectionLoss(nn.Module):
    def __init__(self, num_classes, box_weight=7.5, cls_weight=1.0,
                 focal_gamma=2.0, focal_alpha=0.25):
        super().__init__()
        self.num_classes = num_classes
        self.box_weight = box_weight
        self.cls_weight = cls_weight
        self.focal_gamma = focal_gamma
        self.focal_alpha = focal_alpha
        self.bce = nn.BCEWithLogitsLoss(reduction="none")

    def _focal_loss(self, logits, targets):
        bce = self.bce(logits, targets)
        p = logits.sigmoid()
        p_t = p * targets + (1 - p) * (1 - targets)
        modulating = (1 - p_t) ** self.focal_gamma
        alpha_t = self.focal_alpha * targets + (1 - self.focal_alpha) * (1 - targets)
        return alpha_t * modulating * bce

    def forward(self, preds, gt_boxes_list, gt_classes_list):
        cls_logits = preds["cls_logits"]
        pred_boxes = preds["boxes"]
        anchors = preds["anchors"][0]
        strides = preds["strides"]

        B = cls_logits.shape[0]
        total_cls_loss, total_box_loss = 0.0, 0.0

        for b in range(B):
            gt_boxes = gt_boxes_list[b].to(cls_logits.device)
            gt_classes = gt_classes_list[b].to(cls_logits.device)

            target_cls, target_box, pos_mask = assign_targets_simple(
                pred_boxes[b], anchors, gt_boxes, gt_classes,
                self.num_classes, stride_tensor=strides[0]
            )
            n_pos = max(pos_mask.sum().item(), 1)

            cls_target_onehot = torch.zeros_like(cls_logits[b])
            if pos_mask.any():
                cls_target_onehot[pos_mask, target_cls[pos_mask]] = 1.0

            cls_loss = self._focal_loss(cls_logits[b], cls_target_onehot).sum() / n_pos

            if pos_mask.any():
                box_loss = ciou_loss(pred_boxes[b][pos_mask], target_box[pos_mask]).sum() / n_pos
            else:
                box_loss = torch.tensor(0.0, device=cls_logits.device)

            total_cls_loss += cls_loss
            total_box_loss += box_loss

        total_cls_loss /= B
        total_box_loss /= B
        loss = self.cls_weight * total_cls_loss + self.box_weight * total_box_loss
        return {
            "loss": loss,
            "cls_loss": total_cls_loss.detach(),
            "box_loss": total_box_loss.detach() if torch.is_tensor(total_box_loss) else total_box_loss,
        }


class AuxLosses(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, preds, drivable_target, weather_target, timeofday_target):
        losses = {}
        if "drivable_logits" in preds:
            losses["drivable_loss"] = self.ce(preds["drivable_logits"], drivable_target)
        if "weather_logits" in preds:
            losses["weather_loss"] = self.ce(preds["weather_logits"], weather_target)
        if "timeofday_logits" in preds:
            losses["timeofday_loss"] = self.ce(preds["timeofday_logits"], timeofday_target)
        return losses

In [ ]:
%%writefile /kaggle/working/adas_project/scripts/train_phase1.py
"""
Phase 1 training: object detection (KITTI) + sign location (GTSDB) only.
"""
import argparse
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import sys
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from models.hybrid_model import HybridADASModel
from models.losses import DetectionLoss
from data.phase1_dataset import Phase1Dataset, collate_fn
from configs.phase1_classes import NUM_PHASE1_CLASSES

VARIANT_PRESETS = {
    "cnn_baseline": dict(width_mult=0.50, depth_mult=0.33, num_transformer_layers=0),
    "hybrid_n":     dict(width_mult=0.25, depth_mult=0.33, num_transformer_layers=1),
    "hybrid_s":     dict(width_mult=0.50, depth_mult=0.33, num_transformer_layers=2),
    "hybrid_m":     dict(width_mult=0.75, depth_mult=0.67, num_transformer_layers=4),
}


def build_model(variant):
    cfg = VARIANT_PRESETS[variant]
    return HybridADASModel(
        num_classes=NUM_PHASE1_CLASSES,
        use_drivable_head=False,
        use_weather_head=False,
        **cfg,
    )


def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", required=True)
    ap.add_argument("--variant", default="hybrid_s", choices=list(VARIANT_PRESETS.keys()))
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--img_size", type=int, default=640)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--out_dir", default="/kaggle/working/runs")
    ap.add_argument("--patience", type=int, default=5)
    return ap.parse_args()


def run_epoch(model, loader, loss_fn, optimizer, device, train=True):
    model.train(train)
    total_loss, n_batches = 0.0, 0
    for batch in loader:
        images = batch["image"].to(device)
        boxes_list, classes_list = batch["boxes"], batch["classes"]

        with torch.set_grad_enabled(train):
            preds = model(images)
            losses = loss_fn(preds, boxes_list, classes_list)
            loss = losses["loss"]

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
                optimizer.step()

        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


def main():
    args = parse_args()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(args.out_dir) / f"phase1_{args.variant}"
    out_dir.mkdir(parents=True, exist_ok=True)

    train_ds = Phase1Dataset(args.data_root, split="train", img_size=args.img_size)
    val_ds = Phase1Dataset(args.data_root, split="val", img_size=args.img_size)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                               num_workers=4, collate_fn=collate_fn, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                             num_workers=4, collate_fn=collate_fn)
    print(f"train images: {len(train_ds)}  val images: {len(val_ds)}")

    model = build_model(args.variant).to(device)
    print(f"{args.variant}: {model.count_params()/1e6:.2f}M params, {NUM_PHASE1_CLASSES} classes")

    loss_fn = DetectionLoss(num_classes=NUM_PHASE1_CLASSES)
    optimizer = AdamW(model.parameters(), lr=args.lr, weight_decay=5e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=args.epochs)

    best_val = float("inf")
    epochs_since_improvement = 0
    for epoch in range(args.epochs):
        t0 = time.time()
        train_loss = run_epoch(model, train_loader, loss_fn, optimizer, device, train=True)
        val_loss = run_epoch(model, val_loader, loss_fn, optimizer, device, train=False)
        scheduler.step()
        print(f"[phase1_{args.variant}] epoch {epoch+1}/{args.epochs} "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} ({time.time()-t0:.1f}s)")

        if val_loss < best_val:
            best_val = val_loss
            epochs_since_improvement = 0
            torch.save(model.state_dict(), out_dir / "best.pt")
        else:
            epochs_since_improvement += 1

        torch.save(model.state_dict(), out_dir / "last.pt")

        if epochs_since_improvement >= args.patience:
            print(f"No val improvement for {args.patience} epochs — stopping early "
                  f"at epoch {epoch+1}. Best checkpoint (val_loss={best_val:.4f}) is already saved.")
            break

    print(f"Done. Best val loss {best_val:.4f}, checkpoints in {out_dir}")


if __name__ == "__main__":
    main()

In [ ]:
!grep -n "focal" models/losses.py | head -2
!grep -n "jpg" data/phase1_dataset.py | head -2
!grep -n "patience" scripts/train_phase1.py | head -2

In [ ]:
!pip install -q requests tqdm opencv-python-headless

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
NDJSON_PATH = "/kaggle/input/datasets/ilikesomeone/kitti-ndjson/kitti.ndjson"

!python scripts/convert_kitti_ndjson_to_yolo.py \
    --ndjson_path {NDJSON_PATH} \
    --out_root /kaggle/working/data/phase1_yolo/kitti \
    --max_workers 16

In [ ]:
GTSDB_ROOT = "/kaggle/input/datasets/ilikesomeone/gtsdb-dataset/FullIJCNN2013"

!python scripts/convert_gtsdb_to_yolo.py \
    --gtsdb_root {GTSDB_ROOT} \
    --out_root /kaggle/working/data/phase1_yolo/gtsdb \
    --val_fraction 0.15

In [ ]:
!python scripts/merge_phase1.py \
    --phase1_root /kaggle/working/data/phase1_yolo \
    --kitti_images /kaggle/working/data/phase1_yolo/kitti/images_all \
    --gtsdb_images {GTSDB_ROOT} \
    --out_name merged

In [ ]:
!python scripts/train_phase1.py \
    --data_root /kaggle/working/data/phase1_yolo/merged \
    --variant hybrid_s --epochs 50 --batch_size 16 --patience 10 --lr 5e-5

In [ ]:
import torch, cv2, glob
from models.hybrid_model import HybridADASModel

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HybridADASModel(num_classes=6, use_drivable_head=False, use_weather_head=False,
                         width_mult=0.50, depth_mult=0.33, num_transformer_layers=2).to(device)
model.load_state_dict(torch.load("/kaggle/working/runs/phase1_hybrid_s/best.pt", map_location=device))
model.eval()

def check(img_path):
    img = cv2.imread(img_path)
    img_rgb = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (640, 640))
    img_t = torch.from_numpy(img_rgb).permute(2,0,1).float().unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        preds = model(img_t)
        scores = preds["cls_logits"][0].sigmoid()
    return scores.max().item(), (scores.max(-1).values > 0.35).sum().item()

kitti_samples = glob.glob("/kaggle/working/data/phase1_yolo/merged/images/train/kitti__*")[:5]
gtsdb_samples = glob.glob("/kaggle/working/data/phase1_yolo/merged/images/train/gtsdb__*")[:5]

print("--- KITTI training images ---")
for p in kitti_samples:
    conf, n = check(p)
    print(f"{p.split('/')[-1][:40]}: max_conf={conf:.4f}, boxes_above_0.35={n}")

print("--- GTSDB training images ---")
for p in gtsdb_samples:
    conf, n = check(p)
    print(f"{p.split('/')[-1][:40]}: max_conf={conf:.4f}, boxes_above_0.35={n}")

In [ ]:
import cv2, glob

val_images = sorted(glob.glob("/kaggle/working/data/phase1_yolo/merged/images/val/kitti__*"))[:100]
print(f"Found {len(val_images)} KITTI val images")

first = cv2.imread(val_images[0])
h, w = first.shape[:2]
writer = cv2.VideoWriter("/kaggle/working/kitti_val_test.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h))
for img_path in val_images:
    img = cv2.imread(img_path)
    img = cv2.resize(img, (w, h))
    writer.write(img)
writer.release()
print("Built /kaggle/working/kitti_val_test.mp4")

In [ ]:
!python scripts/infer_phase1_visual.py \
    --video /kaggle/working/kitti_val_test.mp4 \
    --weights /kaggle/working/runs/phase1_hybrid_s/best.pt \
    --variant hybrid_s \
    --out /kaggle/working/kitti_val_annotated.mp4 \
    --max_frames 100

In [ ]:
%run scripts/preview_video_kaggle.py